In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer

In [3]:
df = pd.read_csv('./Dataset/airbus_crash_passengers.csv')

In [5]:
df.sample(5)

,Unnamed: 0,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
225,225,226,0,1,"Jones, James",male,43.1,2,0,66822L,77.37,NaN,C
9,9,10,0,3,"Williams, James",male,36.5,0,2,82692J,174.51,B91,S
410,410,411,0,2,"Brown, James",male,38.5,0,2,69682K,144.88,C52,Q
470,470,471,0,1,"Williams, Mary",female,26.7,3,3,42780O,132.54,NaN,Q
136,136,137,1,2,"Smith, Linda",female,20.6,3,1,30861X,84.05,NaN,Q


In [7]:
df.isnull().sum()

Unnamed: 0       0
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          147
Embarked         0
dtype: int64

In [9]:
# df.drop(columns=[col for col in [df.columns[0], 'PassengerId', 'Name', 'Ticket', 'Cabin'] if col in df.columns], inplace=True)
df.drop(columns=[df.columns[0], 'PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(df.drop('Survived', axis = 1),
                                                   df['Survived'],
                                                   test_size = 0.2,
                                                   random_state = 0)

In [13]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
107,1,female,32.1,1,0,28.15,S
336,3,female,20.1,4,3,52.17,Q
71,2,female,48.5,1,2,42.53,C
474,3,male,49.7,3,3,83.96,C
6,2,male,49.0,1,2,37.88,S


In [15]:
y_train.head()

107    1
336    1
71     0
474    1
6      1
Name: Survived, dtype: int64

In [17]:
df.isnull().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

In [19]:
# Applying Simple imputing
# Give number of the column instead of Nme, like Age comes on second index and Embarked on 6th one.
trnf1 = ColumnTransformer([
    ('impute_age', SimpleImputer(), [2]),
    ('impute_emb', SimpleImputer(strategy = 'most_frequent'), [6])
], remainder = 'passthrough')

In [21]:
# Applying OneHot Encoding
trnf2 = ColumnTransformer([
    ('ohe_sex_emb',OneHotEncoder(sparse_output = False, handle_unknown = 'ignore'),[1,6])
], remainder = 'passthrough')

In [23]:
# Applying Feature Scaling
trnf3 = ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0,10))
])

In [25]:
# Applying Feature Selection
trnf4 = SelectKBest(score_func = chi2, k = 8)

In [27]:
# training the model now
trnf5 = DecisionTreeClassifier()

In [29]:
pipe = Pipeline([
    ('trnf1', trnf1),
    ('trnf2', trnf2),
    ('trnf3', trnf3),
    ('trnf4', trnf4),
    ('trnf5', trnf5)
])

In [31]:
# Alernate Use:
# pipe = make_pipeline(trnf1,trnf2,trnf3,trnf4,trnf5)

In [32]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('trnf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_emb',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trnf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_emb',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('trnf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trnf4',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x000002BE3C1B9080>)),
                ('trnf5', DecisionTreeClassifier())])

In [343]:
 pipe.named_steps['trnf1']

ColumnTransformer(remainder='passthrough',
                  transformers=[('impute_age', SimpleImputer(), [2]),
                                ('impute_emb',
                                 SimpleImputer(strategy='most_frequent'),
                                 [6])])

In [345]:
pipe

Pipeline(steps=[('trnf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_emb',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trnf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_emb',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('trnf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trnf4',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x000001BBA2D6C9A0>)),
                ('trnf5', DecisionTreeClassifier())])

In [347]:
y_pred = pipe.predict(X_test)

In [349]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.51

In [351]:
# cross validation using cross_val_score
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

0.49250000000000005

In [363]:
# gridsearchcv
params = {'trnf5__max_depth':[1,2,3,4,5,None]}

In [365]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('trnf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('impute_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('impute_emb',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [6])])),
                                       ('trnf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe_sex_emb',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         [1,
                                                                          6])])),
                                       ('trnf3',
                                        ColumnTransformer(transformers=[('scale',
                                                                         MinMaxScaler(),
                                                                         slice(0, 10, None))])),
                                       ('trnf4',
                                        SelectKBest(k=8,
                                                    score_func=<function chi2 at 0x000001BBA2D6C9A0>)),
                                       ('trnf5', DecisionTreeClassifier())]),
             param_grid={'trnf5__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [366]:
grid.best_score_

0.49250000000000005

In [369]:
# export
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))